# Simple House Price Prediction
**Machine Learning CIE 2 – Activity 2**  
**Unit II – Supervised Learning: Regression**

## Objective
Build a Linear Regression model to predict house prices using a numerical property feature.

> Dataset: Download `Housing.csv` from [Kaggle – Housing Prices Dataset](https://www.kaggle.com/datasets/yasserh/housing-prices-dataset), then upload it when prompted below. Run every cell in order. Use the actual outputs from your run in your submission.

## Step 1: Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Step 2: Upload and load the dataset

In [ ]:
# Google Colab upload
from google.colab import files
uploaded = files.upload()

# Change the filename here if your CSV has a different name
df = pd.read_csv("Housing.csv")
df.head()

## Step 3: Inspect and clean the data

In [ ]:
print("Shape (rows, columns):", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nSummary statistics:\n", df.describe(include="all"))
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# Remove exact duplicate rows
df = df.drop_duplicates()

# Convert selected columns to numeric and remove rows with invalid/missing values
df["area"] = pd.to_numeric(df["area"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["area", "price"])
df = df[(df["area"] > 0) & (df["price"] > 0)]

print("\nCleaned dataset shape:", df.shape)

## Step 4: Select input feature (X) and target (y)
We use **area** as the input feature because larger houses may generally have higher prices. We use **price** as the continuous numerical target. We do not use an ID column as a predictive feature.

In [ ]:
X = df[["area"]]
y = df["price"]

print("Input feature:", X.columns.tolist())
print("Target:", y.name)

## Step 5: Visualise area and actual price

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["area"], df["price"], alpha=0.6)
plt.title("House Area vs House Price")
plt.xlabel("Area")
plt.ylabel("Price")
plt.grid(True)
plt.show()

**Interpretation:** Observe the scatter plot. If points generally rise from left to right, the relationship is positive. A visible relationship does not by itself prove that area causes the price to change.

## Step 6: Split into training and testing data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

The training data is used to learn the model parameters. The test data is kept separate and is used to evaluate the model on unseen observations.

## Step 7: Train Linear Regression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

coefficient = model.coef_[0]
intercept = model.intercept_

print("Coefficient (slope):", coefficient)
print("Intercept:", intercept)
print(f"Model equation: Predicted price = {intercept:.2f} + ({coefficient:.2f} × area)")

**Meaning:** The coefficient (slope) indicates the estimated change in predicted price for one additional unit of area. The intercept is the model's predicted price when area is zero; it may not have a practical meaning for a real house.

## Step 8: Predict test-set prices

In [ ]:
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "Area": X_test["area"].values,
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})
comparison.head(10)

Compare the actual and predicted prices above. Differences are prediction errors. These are predictions for the test set, not the training set.

## Step 9: Evaluate the model

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R² score:", r2)

**Metric meanings:**
- **MAE:** Average absolute difference between actual and predicted prices.
- **MSE:** Average squared prediction error; large errors receive more weight.
- **RMSE:** Square root of MSE, expressed in the same units as price.
- **R²:** Compares the model with a baseline that predicts the test-set mean. It can be negative on test data.

Interpret the values using the price units in the dataset. Smaller MAE/RMSE generally mean smaller errors. R² closer to 1 generally indicates a better fit.

## Step 10: Actual vs predicted plot

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
lower = min(y_test.min(), y_pred.min())
upper = max(y_test.max(), y_pred.max())
plt.plot([lower, upper], [lower, upper], "r--", label="Perfect prediction")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted House Prices")
plt.legend()
plt.grid(True)
plt.show()

Points closer to the red reference line represent more accurate predictions. Points far from the line indicate larger errors.

## Step 11: Predict a new example

In [ ]:
# Choose an area within the range of the training data
new_area = float(X_train["area"].median())
new_house = pd.DataFrame({"area": [new_area]})
new_prediction = model.predict(new_house)[0]

print("New house area:", new_area)
print("Predicted house price:", new_prediction)
print("Price units: same units as the dataset's price column")

## Conclusion
A Linear Regression model was trained to predict house price using area as the input feature. The dataset was inspected, cleaned for missing/invalid area and price values, and split into training and testing sets. The model was evaluated using MAE, MSE, RMSE and R², and actual prices were compared with predicted prices.

**Limitation:** House prices also depend on location, number of bedrooms, age, condition and other factors. A one-feature linear model may not capture all these factors and should not be treated as a professional property valuation.